# Intrusion Detection System — Enhanced Notebook
**Dataset:** NSL-KDD | **Model:** Random Forest | **Goal:** Binary classification (normal vs anomaly)

This notebook covers: data loading → preprocessing → EDA → model training → evaluation (with ROC-AUC, F1, confusion matrix) → SHAP explainability → model export.

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import sklearn
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)

print(f'sklearn version: {sklearn.__version__}  ← pin this in requirements.txt')
print(f'pandas  version: {pd.__version__}')
print(f'numpy   version: {np.__version__}')

## 1 — Data Loading & Validation

In [ ]:
# ── Cell 2: Load data (relative paths — works on any machine) ────────────────
# Place Train_data.csv and Test_data.csv in the same folder as this notebook
train_df = pd.read_csv('Train_data.csv')
test_df  = pd.read_csv('Test_data.csv')

print('Train shape:', train_df.shape)
print('Test  shape:', test_df.shape)

# Sanity checks
assert train_df.isnull().sum().sum() == 0, 'Missing values in train!'
assert test_df.isnull().sum().sum()  == 0, 'Missing values in test!'
print('No missing values confirmed.')

print('\nClass distribution (train):')
print(train_df['class'].value_counts())
print(f'\nClass balance: {train_df["class"].value_counts(normalize=True).round(3).to_dict()}')

## 2 — Preprocessing

In [ ]:
# ── Cell 3: Label-encode categorical columns + save encoders ─────────────────
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']

train_encoded = train_df.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    train_encoded[col] = le.fit_transform(train_encoded[col])
    joblib.dump(le, f'{col}_classes.pkl')
    print(f'{col}: {len(le.classes_)} unique values → saved {col}_classes.pkl')

# Encode the target
X = train_encoded.drop('class', axis=1)
y = train_encoded['class']  # keep as strings ('normal'/'anomaly') — RF handles it

print(f'\nFeature matrix: {X.shape}')
print('Features:', list(X.columns))

## 3 — Exploratory Data Analysis

In [ ]:
# ── Cell 4: Class distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train_df['class'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2196F3','#F44336'], edgecolor='white', linewidth=0.5)
axes[0].set_title('Class Distribution — Count')
axes[0].set_ylabel('Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2196F3','#F44336'], startangle=90)
axes[1].set_title('Class Distribution — Proportion')

plt.suptitle('NSL-KDD Target Variable', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Feature correlation heatmap ─────────────────────────────────────
# Use top 15 most-varying numeric features for readability
numeric_cols = X.select_dtypes(include=[np.number]).columns
top_var_cols = X[numeric_cols].var().nlargest(15).index

fig, ax = plt.subplots(figsize=(12, 10))
corr = X[top_var_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.4, ax=ax, square=True)
ax.set_title('Correlation Heatmap — Top 15 High-Variance Features', fontsize=13)
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: Key feature distributions by class ───────────────────────────────
key_features = ['src_bytes', 'dst_bytes', 'count', 'srv_count',
                'same_srv_rate', 'diff_srv_rate']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

plot_df = X[key_features].copy()
plot_df['class'] = train_df['class'].values

for i, feat in enumerate(key_features):
    for cls, color in [('normal','#2196F3'), ('anomaly','#F44336')]:
        vals = plot_df[plot_df['class']==cls][feat]
        # Log scale for highly skewed features
        vals_log = np.log1p(vals)
        axes[i].hist(vals_log, bins=40, alpha=0.6, color=color, label=cls, edgecolor='none')
    axes[i].set_title(f'log(1+{feat})')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions: Normal vs Anomaly (log scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: Statistical test (t-test on src_bytes) ──────────────────────────
from scipy.stats import ttest_ind, mannwhitneyu

normal_bytes  = train_df[train_df['class']=='normal']['src_bytes']
anomaly_bytes = train_df[train_df['class']=='anomaly']['src_bytes']

t_stat, p_val    = ttest_ind(normal_bytes, anomaly_bytes)
u_stat, p_val_mw = mannwhitneyu(normal_bytes, anomaly_bytes, alternative='two-sided')

print('=== Statistical Tests: src_bytes (Normal vs Anomaly) ===')
print(f"T-test      : t={t_stat:.2f}, p={p_val:.2e}")
print(f"Mann-Whitney: U={u_stat:.0f}, p={p_val_mw:.2e}")
print()
if p_val < 0.05:
    print('Result: Statistically significant difference (p < 0.05)')
    print('Implication: src_bytes is a useful discriminating feature')

## 4 — Model Training

In [ ]:
# ── Cell 8: Train/Validation split — STRATIFIED ──────────────────────────────
# stratify=y ensures both splits have the same class ratio as the full dataset
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ← IMPORTANT: missing in original code
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print('Train class ratio:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Val   class ratio:', y_val.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ── Cell 9: Train Random Forest ──────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,           # use all CPU cores
    class_weight='balanced'  # handles slight class imbalance
)

model.fit(X_train, y_train)
print('Training complete.')
print(f'sklearn version used: {sklearn.__version__}')

In [ ]:
# ── Cell 10: 5-Fold Cross-Validation ─────────────────────────────────────────
# Cross-validation gives a more reliable accuracy estimate than a single split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)

print('5-Fold Cross-Validation Accuracy:')
for i, s in enumerate(cv_scores):
    print(f'  Fold {i+1}: {s*100:.4f}%')
print(f'  Mean:   {cv_scores.mean()*100:.4f}% ± {cv_scores.std()*100:.4f}%')

## 5 — Evaluation (Full Metrics Suite)

In [ ]:
# ── Cell 11: Core metrics ────────────────────────────────────────────────────
y_pred  = model.predict(X_val)
y_proba = model.predict_proba(X_val)  # shape (n, 2) — columns: ['anomaly', 'normal']

# anomaly is classes_[0], so anomaly probability = column 0
anomaly_idx = list(model.classes_).index('anomaly')
y_proba_anomaly = y_proba[:, anomaly_idx]
y_val_binary    = (y_val == 'anomaly').astype(int)

accuracy  = accuracy_score(y_val, y_pred)
f1        = f1_score(y_val, y_pred, pos_label='anomaly')
roc_auc   = roc_auc_score(y_val_binary, y_proba_anomaly)

print('=== VALIDATION METRICS ===')
print(f'Accuracy  : {accuracy*100:.4f}%')
print(f'F1-Score  : {f1:.6f}')
print(f'ROC-AUC   : {roc_auc:.6f}')
print()
print('Classification Report:')
print(classification_report(y_val, y_pred))

In [ ]:
# ── Cell 12: Confusion Matrix with rates ─────────────────────────────────────
cm = confusion_matrix(y_val, y_pred, labels=['normal','anomaly'])
tn, fp, fn, tp = cm.ravel()

fpr_rate = fp / (fp + tn) * 100  # false positive rate — alerts on normal traffic
fnr_rate = fn / (fn + tp) * 100  # false negative rate — missed attacks

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Anomaly'],
            yticklabels=['Normal','Anomaly'], ax=axes[0])
axes[0].set_title('Confusion Matrix — Raw Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalised (percentage)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['Normal','Anomaly'],
            yticklabels=['Normal','Anomaly'], ax=axes[1])
axes[1].set_title('Confusion Matrix — Normalised (%)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle(f'False Positive Rate: {fpr_rate:.2f}%  |  False Negative Rate (Missed Attacks): {fnr_rate:.2f}%',
             fontsize=11)
plt.tight_layout()
plt.savefig('plot_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'True Negatives  (correct normal)  : {tn}')
print(f'False Positives (normal → anomaly): {fp}  ← alert fatigue')
print(f'False Negatives (anomaly → normal): {fn}  ← missed attacks')
print(f'True Positives  (correct anomaly) : {tp}')

In [ ]:
# ── Cell 13: ROC Curve + Precision-Recall Curve ──────────────────────────────
fpr_arr, tpr_arr, _ = roc_curve(y_val_binary, y_proba_anomaly)
prec, recall, _     = precision_recall_curve(y_val_binary, y_proba_anomaly)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr_arr, tpr_arr, color='#2196F3', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
axes[0].fill_between(fpr_arr, tpr_arr, alpha=0.1, color='#2196F3')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall
axes[1].plot(recall, prec, color='#F44336', lw=2)
axes[1].fill_between(recall, prec, alpha=0.1, color='#F44336')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.savefig('plot_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 14: Feature Importance ──────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=X.columns)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#F44336' if i >= 15 else '#2196F3' for i in range(len(top20))]
top20.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13)
ax.set_xlabel('Mean Decrease in Impurity')
ax.axvline(importances.mean(), color='gray', linestyle='--', alpha=0.7, label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — SHAP Explainability

In [ ]:
# ── Cell 15: SHAP global explanation ─────────────────────────────────────────
# Install if needed: pip install shap
import shap

# Use a background sample for efficiency (200 rows)
background = shap.sample(X_train, 200, random_state=42)
explainer  = shap.TreeExplainer(model, background)

# Explain validation set (use a sample for speed)
X_val_sample = X_val.sample(500, random_state=42)
shap_values  = explainer.shap_values(X_val_sample)

# shap_values is a list: [class_0_shap, class_1_shap]
# anomaly = class index 0 (alphabetical order: ['anomaly','normal'])
shap_anomaly = shap_values[anomaly_idx]

print('SHAP values shape:', shap_anomaly.shape)
print('Computing summary plot...')

In [ ]:
# ── Cell 16: SHAP Summary Plot ───────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_anomaly, X_val_sample,
    feature_names=list(X.columns),
    show=False, plot_type='dot'
)
plt.title('SHAP Summary — Feature Impact on Anomaly Detection', fontsize=13)
plt.tight_layout()
plt.savefig('plot_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Each dot = one prediction. Red = high feature value, Blue = low.')
print('Position on X-axis = impact on prediction (right = pushes toward anomaly)')

In [ ]:
# ── Cell 17: SHAP per-prediction explanation (single record) ─────────────────
# Pick one anomaly and one normal record and explain them
sample_anomaly = X_val[y_val == 'anomaly'].iloc[[0]]
sample_normal  = X_val[y_val == 'normal'].iloc[[0]]

for label, sample in [('Anomaly', sample_anomaly), ('Normal', sample_normal)]:
    sv   = explainer.shap_values(sample)
    sv_a = sv[anomaly_idx][0]
    base = explainer.expected_value[anomaly_idx]
    
    # Top 5 contributing features
    contrib = pd.Series(sv_a, index=X.columns).abs().nlargest(5)
    print(f'\n=== {label} record — Top 5 SHAP features ===')
    for feat in contrib.index:
        direction = 'TOWARD anomaly' if sv_a[list(X.columns).index(feat)] > 0 else 'AWAY from anomaly'
        print(f'  {feat:<35} value={sample[feat].values[0]:<10} impact={sv_a[list(X.columns).index(feat)]:+.4f} ({direction})')

## 7 — Export Model & Artifacts

In [ ]:
# ── Cell 18: Save model + metadata ───────────────────────────────────────────
import json, datetime

joblib.dump(model, 'ids_model.pkl')
print('Model saved: ids_model.pkl')

# Save model metadata so the app can display it without reloading the model
metadata = {
    'sklearn_version'   : sklearn.__version__,
    'trained_at'        : datetime.datetime.now().isoformat(),
    'n_estimators'      : model.n_estimators,
    'n_features'        : model.n_features_in_,
    'feature_names'     : list(X.columns),
    'classes'           : list(model.classes_),
    'accuracy'          : round(accuracy_score(y_val, model.predict(X_val)), 6),
    'f1_anomaly'        : round(f1_score(y_val, model.predict(X_val), pos_label='anomaly'), 6),
    'roc_auc'           : round(roc_auc, 6),
    'top_features'      : importances.nlargest(10).index.tolist(),
    'confusion_matrix'  : cm.tolist(),
    'false_positive_rate': round(fpr_rate, 4),
    'false_negative_rate': round(fnr_rate, 4),
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadata saved: model_metadata.json')
print(json.dumps(metadata, indent=2))

In [ ]:
# ── Cell 19: Apply model to test set + save predictions ──────────────────────
test_encoded = test_df.copy()

for col in CATEGORICAL_COLS:
    le = joblib.load(f'{col}_classes.pkl')
    # Map unknown categories to the most frequent known class (safe fallback)
    test_encoded[col] = test_encoded[col].apply(
        lambda x: x if x in le.classes_ else le.classes_[0]
    )
    test_encoded[col] = le.transform(test_encoded[col])

y_test_pred  = model.predict(test_encoded)
y_test_proba = model.predict_proba(test_encoded)[:, anomaly_idx]

predictions_df = pd.DataFrame({
    'prediction'       : y_test_pred,
    'anomaly_probability': np.round(y_test_proba, 4)
})

predictions_df.to_csv('predictions.csv', index=False)
print('Predictions saved: predictions.csv')
print(predictions_df['prediction'].value_counts())
print(predictions_df.head(10))